In [ ]:
# Optimization of genetic algorithm parameter in hybrid genetic algorithm-neural network modelling
# Application to spray drying of coconut milk.

In [2]:
import random
import numpy as np
from deap import base, creator, tools, algorithms
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load Iris Dataset
iris = load_iris()
X = iris.data
y = iris.target

# Normalize
scaler = MinMaxScaler()
X = scaler.fit_transform(X)

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

# GA Setup
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# Attributes (genes)
toolbox.register("attr_hidden", random.randint, 5, 20)
toolbox.register("attr_lr", random.uniform, 0.001, 0.1)
toolbox.register("attr_alpha", random.uniform, 0.0001, 0.01)

# Individual = [hidden_neurons, learning_rate, alpha]
toolbox.register(
    "individual",
    tools.initCycle,
    creator.Individual,
    (toolbox.attr_hidden,
     toolbox.attr_lr,
     toolbox.attr_alpha),
    n=1
)

toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Fitness Function
def evaluate(individual):
    hidden = int(max(5, min(20, round(individual[0]))))
    lr = max(0.0001, individual[1])
    alpha = max(0.00001, individual[2])

    model = MLPClassifier(
        hidden_layer_sizes=(hidden,),
        learning_rate_init=lr,
        alpha=alpha,
        max_iter=1000,
        random_state=1
    )

    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)  # accuracy

    return (score,)

toolbox.register("evaluate", evaluate)
toolbox.register("mate", tools.cxTwoPoint)

toolbox.register(
    "mutate",
    tools.mutGaussian,
    mu=0,
    sigma=0.1,
    indpb=0.2
)

toolbox.register(
    "select",
    tools.selTournament,
    tournsize=3
)

# Run GA
if __name__ == "__main__":
    pop = toolbox.population(n=20)

    algorithms.eaSimple(
        pop,
        toolbox,
        cxpb=0.7,
        mutpb=0.2,
        ngen=50,
        verbose=False
    )

    best = tools.selBest(pop, 1)[0]

    print("Best Parameters")
    print("Hidden Neurons:", int(best[0]))
    print("Learning Rate:", round(best[1], 4))
    print("Alpha:", round(best[2], 5))

    # Final Model
    final_hidden = int(round(best[0]))
    final_lr = max(0.0001, best[1])
    final_alpha = max(0.00001, best[2])

    final_model = MLPClassifier(
        hidden_layer_sizes=(final_hidden,),
        learning_rate_init=final_lr,
        alpha=final_alpha,
        max_iter=1000,
        random_state=1
    )

    final_model.fit(X_train, y_train)

    # Predictions
    pred = final_model.predict(X_test)

    print("\nPredictions (Sample)")
    for i in range(min(10, len(y_test))):
        print("Actual:", y_test[i], "Predicted:", pred[i])

C:\Users\Aarya\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\Aarya\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\Aarya\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\Aarya\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\Users\Aarya\anaconda3\Lib\site-packages\sklearn\neural_network\_m

Best Parameters
Hidden Neurons: 5
Learning Rate: 0.0808
Alpha: -0.21684

Predictions (Sample)
Actual: 0 Predicted: 0
Actual: 1 Predicted: 1
Actual: 1 Predicted: 1
Actual: 0 Predicted: 0
Actual: 2 Predicted: 2
Actual: 1 Predicted: 1
Actual: 2 Predicted: 2
Actual: 0 Predicted: 0
Actual: 0 Predicted: 0
Actual: 2 Predicted: 2


In [1]:
!pip install deap